## Setup

First, let's set up the Python environment and import necessary libraries.

In [1]:
# Configure loguru FIRST to suppress verbose logging in notebooks only
import sys
import os
from loguru import logger
logger.remove()  # Remove default handler

# Add filter to suppress tech_adoption warnings
def filter_tech_warnings(record):
    return not any(msg in record["message"] for msg in [
        "No data found for technology",
        "No growth data available"
    ])

logger.add(sys.stderr, level='WARNING', format='<level>{level: <8}</level> | <cyan>{name}</cyan>:<cyan>{function}</cyan>:<cyan>{line}</cyan> - <level>{message}</level>', filter=filter_tech_warnings)

from pathlib import Path

# Get the BICEP root directory (three levels up from this notebook)
bicep_root = Path.cwd().parent.parent.parent
if str(bicep_root) not in sys.path:
    sys.path.insert(0, str(bicep_root))

import pandas as pd
import numpy as np

from bicep.analysis import BicepResults, BicepMultiStateResults

print("Environment setup complete!")

Environment setup complete!


In [2]:
import os

# SUPPRESS LOGGING FOR CLEAN NOTEBOOK OUTPUT
# To see full BICEP logging details, comment out the line below:
os.environ['LOGURU_LEVEL'] = 'WARNING'

## Scenario Overview

### BAU (Business As Usual)
Reflects current policies and baseline technology adoption trends.
- Moderate technology adoption rates
- Lower overall electrical load growth

### High (High Demand Growth)
Assumes higher technology adoption and increased demand.
- Higher technology adoption rates
- Significant increase in electrical load
- Greater infrastructure upgrade requirements

Let's run analyses for both scenarios and compare the results.

In [3]:
# Run both scenarios from SQLite database
print("Running BAU scenario analysis for all US states...")
bau_all = BicepMultiStateResults(scenario='bau', mode='local', target_states='all')
bau_results = bau_all.all_states_buildings

print("\nRunning High scenario analysis for all US states...")
high_all = BicepMultiStateResults(scenario='high', mode='local', target_states='all')
high_results = high_all.all_states_buildings

print(f"\nScenarios loaded successfully!")
print(f"  BAU: {len(bau_results):,} buildings")
print(f"  High: {len(high_results):,} buildings")

Running BAU scenario analysis for all US states...


/Users/faye994/code/BICEP/bicep/capacity.py:329: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bldg['represented_vehicles'] = (bldg['total_units']/5).fillna(1) * bldg['total_parking_spaces']
/Users/faye994/code/BICEP/bicep/capacity.py:329: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bldg['represented_vehicles'] = (bldg['total_units']/5).fillna(1) * bldg['total_parking_spaces']



Running High scenario analysis for all US states...


/Users/faye994/code/BICEP/bicep/capacity.py:329: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bldg['represented_vehicles'] = (bldg['total_units']/5).fillna(1) * bldg['total_parking_spaces']
/Users/faye994/code/BICEP/bicep/capacity.py:329: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bldg['represented_vehicles'] = (bldg['total_units']/5).fillna(1) * bldg['total_parking_spaces']



Scenarios loaded successfully!
  BAU: 884,596 buildings
  High: 884,596 buildings


## Total Cost Comparison

In [4]:
# Calculate total costs (using weighted_cost which reflects infrastructure upgrade costs)
bau_total = bau_results['weighted_cost'].sum()
high_total = high_results['weighted_cost'].sum()
difference = high_total - bau_total
percent_increase = (difference / bau_total) * 100

# Residential vs Commercial
bau_res = bau_results[bau_results['residential'] == 1]['weighted_cost'].sum()
bau_com = bau_results[bau_results['residential'] == 0]['weighted_cost'].sum()
high_res = high_results[high_results['residential'] == 1]['weighted_cost'].sum()
high_com = high_results[high_results['residential'] == 0]['weighted_cost'].sum()

print("="*70)
print("TOTAL ANNUALIZED INFRASTRUCTURE UPGRADE COSTS (2020-2050)")
print("="*70)
print(f"\nBAU Scenario:")
print(f"  Total:           ${bau_total:>20,.0f}")
print(f"  Residential:     ${bau_res:>20,.0f}")
print(f"  Commercial:      ${bau_com:>20,.0f}")

print(f"\nHigh Scenario:")
print(f"  Total:           ${high_total:>20,.0f}")
print(f"  Residential:     ${high_res:>20,.0f}")
print(f"  Commercial:      ${high_com:>20,.0f}")

print(f"\nDifference (High - BAU):")
print(f"  Absolute:        ${difference:>20,.0f}")
print(f"  Percent:         {percent_increase:>20.1f}%")
print("="*70)

TOTAL ANNUALIZED INFRASTRUCTURE UPGRADE COSTS (2020-2050)

BAU Scenario:
  Total:           $       5,906,123,290
  Residential:     $       5,294,950,698
  Commercial:      $         611,172,592

High Scenario:
  Total:           $       8,699,693,648
  Residential:     $       7,878,916,817
  Commercial:      $         820,776,831

Difference (High - BAU):
  Absolute:        $       2,793,570,358
  Percent:                         47.3%


## Cost Distribution by Building Type

In [5]:
# Create comparison chart (text-based to avoid rendering issues)
scenarios = ['BAU', 'High']
residential_costs = [bau_res, high_res]
commercial_costs = [bau_com, high_com]

print("\nComparison by Building Type:")
print(f"{'Scenario':<10} {'Residential':>20} {'Commercial':>20} {'Total':>20}")
print("-" * 72)
for i, scenario in enumerate(scenarios):
    total = residential_costs[i] + commercial_costs[i]
    print(f"{scenario:<10} ${residential_costs[i]:>18,.0f} ${commercial_costs[i]:>18,.0f} ${total:>18,.0f}")


Comparison by Building Type:
Scenario            Residential           Commercial                Total
------------------------------------------------------------------------
BAU        $     5,294,950,698 $       611,172,592 $     5,906,123,290
High       $     7,878,916,817 $       820,776,831 $     8,699,693,648


## Cost Drivers Comparison

Let's visualize how different technologies drive costs in each scenario.

In [6]:
# Analyze cost distribution and drivers
def analyze_cost_distribution(df, scenario_name):
    """Analyze cost distribution and drivers"""
    
    total_buildings = len(df)
    buildings_upgraded = (df['upgrade_required'] == 1).sum()
    
    print(f"\n{scenario_name} - Cost Distribution Analysis:")
    print(f"  Total buildings: {total_buildings:,}")
    print(f"  Buildings requiring upgrades: {buildings_upgraded:,} ({buildings_upgraded/total_buildings*100:.1f}%)")
    print(f"  Total cost: ${df['weighted_cost'].sum():,.0f}")
    print(f"  Mean cost (all): ${df['weighted_cost'].mean():,.0f}")
    print(f"  Mean cost (upgraded only): ${df[df['weighted_cost'] > 0]['weighted_cost'].mean():,.0f}")
    
    # State distribution
    top_states = df.groupby('state')['weighted_cost'].sum().nlargest(5)
    print(f"  Top 5 states by cost:")
    for state, cost in top_states.items():
        print(f"    {state}: ${cost:,.0f}")

analyze_cost_distribution(bau_results, "BAU")
analyze_cost_distribution(high_results, "High")


BAU - Cost Distribution Analysis:
  Total buildings: 884,596
  Buildings requiring upgrades: 119,278 (13.5%)
  Total cost: $5,906,123,290
  Mean cost (all): $49,516
  Mean cost (upgraded only): $49,516
  Top 5 states by cost:
    CA: $1,043,882,459
    TX: $383,017,640
    FL: $277,208,268
    MI: $266,026,812
    NY: $226,466,378

High - Cost Distribution Analysis:
  Total buildings: 884,596
  Buildings requiring upgrades: 172,948 (19.6%)
  Total cost: $8,699,693,648
  Mean cost (all): $50,302
  Mean cost (upgraded only): $50,302
  Top 5 states by cost:
    CA: $1,080,650,138
    TX: $504,093,793
    MI: $502,301,062
    NY: $404,686,189
    IL: $382,508,789


In [7]:
# Analyze residential vs commercial breakdown
print("\n" + "="*70)
print("BUILDING TYPE BREAKDOWN")
print("="*70)

for scenario_name, df in [("BAU", bau_results), ("High", high_results)]:
    res_count = (df['residential'] == 1).sum()
    com_count = (df['residential'] == 0).sum()
    res_cost = df[df['residential'] == 1]['weighted_cost'].sum()
    com_cost = df[df['residential'] == 0]['weighted_cost'].sum()
    
    print(f"\n{scenario_name}:")
    print(f"  Residential: {res_count:,} buildings, ${res_cost:,.0f} total cost")
    print(f"  Commercial: {com_count:,} buildings, ${com_cost:,.0f} total cost")


BUILDING TYPE BREAKDOWN

BAU:
  Residential: 548,916 buildings, $5,294,950,698 total cost
  Commercial: 335,680 buildings, $611,172,592 total cost

High:
  Residential: 548,916 buildings, $7,878,916,817 total cost
  Commercial: 335,680 buildings, $820,776,831 total cost


## Costs by Year

In [8]:
# Get costs by state (top 10)
bau_by_state = bau_results.groupby('state')['weighted_cost'].sum().sort_values(ascending=False).head(10)
high_by_state = high_results.groupby('state')['weighted_cost'].sum().sort_values(ascending=False).head(10)

# Create comparison - use intersection of states to avoid KeyError
common_states = list(set(bau_by_state.index) & set(high_by_state.index))
comparison_states = pd.DataFrame({
    'State': bau_by_state[common_states].index,
    'BAU': bau_by_state[common_states].values,
    'High': high_by_state[common_states].values
})

comparison_states['Difference'] = comparison_states['High'] - comparison_states['BAU']
comparison_states['Percent Increase'] = (comparison_states['Difference'] / comparison_states['BAU'] * 100).round(1)

print("\nTop 10 States by Total Infrastructure Upgrade Costs:")
print("="*90)
for idx, row in comparison_states.iterrows():
    print(f"{row['State']:<5} BAU: ${row['BAU']:>15,.0f}  |  High: ${row['High']:>15,.0f}  |  Increase: {row['Percent Increase']:>6.1f}%")


Top 10 States by Total Infrastructure Upgrade Costs:
OH    BAU: $    168,691,798  |  High: $    338,507,536  |  Increase:  100.7%
PA    BAU: $    167,685,085  |  High: $    305,363,205  |  Increase:   82.1%
NC    BAU: $    195,397,881  |  High: $    269,496,620  |  Increase:   37.9%
MI    BAU: $    266,026,812  |  High: $    502,301,062  |  Increase:   88.8%
CA    BAU: $  1,043,882,459  |  High: $  1,080,650,138  |  Increase:    3.5%
FL    BAU: $    277,208,268  |  High: $    343,600,779  |  Increase:   24.0%
NJ    BAU: $    204,960,289  |  High: $    278,330,501  |  Increase:   35.8%
NY    BAU: $    226,466,378  |  High: $    404,686,189  |  Increase:   78.7%
TX    BAU: $    383,017,640  |  High: $    504,093,793  |  Increase:   31.6%
IL    BAU: $    190,910,712  |  High: $    382,508,789  |  Increase:  100.4%


## Costs by State (Top 10)

In [9]:
# Analyze regional variation
print("\nRegional Analysis:")
print("-" * 90)
print(f"California (largest):")
ca_bau = bau_results[bau_results['state'] == 'CA']['weighted_cost'].sum()
ca_high = high_results[high_results['state'] == 'CA']['weighted_cost'].sum()
print(f"  BAU: ${ca_bau:,.0f}")
print(f"  High: ${ca_high:,.0f}")
if ca_bau > 0:
    print(f"  Increase: {((ca_high - ca_bau) / ca_bau * 100):.1f}%")


Regional Analysis:
------------------------------------------------------------------------------------------
California (largest):
  BAU: $1,043,882,459
  High: $1,080,650,138
  Increase: 3.5%


In [10]:
# Analyze regional variation
print("\nRegional Analysis:")
print("-" * 90)
print(f"California (largest):")
ca_bau = bau_results[bau_results['state'] == 'CA']['weighted_cost'].sum()
ca_high = high_results[high_results['state'] == 'CA']['weighted_cost'].sum()
print(f"  BAU: ${ca_bau:,.0f}")
print(f"  High: ${ca_high:,.0f}")
print(f"  Increase: {((ca_high - ca_bau) / ca_bau * 100):.1f}%")


Regional Analysis:
------------------------------------------------------------------------------------------
California (largest):
  BAU: $1,043,882,459
  High: $1,080,650,138
  Increase: 3.5%


## Capacity Requirements Comparison

In [11]:
# Compare capacity requirements by analyzing upgrade patterns
print("\n" + "="*70)
print("CAPACITY REQUIREMENT ANALYSIS")
print("="*70)

for scenario_name, df in [("BAU", bau_results), ("High", high_results)]:
    residential = df[df['residential'] == 1]
    
    # Buildings needing upgrades
    upgrades_needed = (residential['upgrade_required'] == 1).sum()
    total_residential = len(residential)
    
    # Capacity metrics (max electrical loads)
    avg_load = residential['max_elec_consumption_kwh'].mean()
    total_load = residential['max_elec_consumption_kwh'].sum()
    
    print(f"\n{scenario_name} - Residential:")
    print(f"  Buildings needing upgrades: {upgrades_needed:,} / {total_residential:,} ({upgrades_needed/total_residential*100:.1f}%)")
    print(f"  Average max electrical load: {avg_load:,.0f} kWh/year")
    print(f"  Total peak load: {total_load/1e6:,.1f} million kWh/year")


CAPACITY REQUIREMENT ANALYSIS

BAU - Residential:
  Buildings needing upgrades: 80,606 / 548,916 (14.7%)
  Average max electrical load: 2 kWh/year
  Total peak load: 1.3 million kWh/year

High - Residential:
  Buildings needing upgrades: 120,865 / 548,916 (22.0%)
  Average max electrical load: 2 kWh/year
  Total peak load: 1.3 million kWh/year


## Key Findings

### Cost Differences
The High scenario requires significantly higher infrastructure investments due to increased technology adoption.

In [12]:
# Summary findings
print("\n" + "="*70)
print("KEY FINDINGS: SCENARIO COMPARISON")
print("="*70)

print(f"\n1. COST IMPACT:")
print(f"   The High scenario requires ${difference:,.0f} more in annual costs")
print(f"   This represents a {percent_increase:.1f}% increase over BAU")

print(f"\n2. RESIDENTIAL VS COMMERCIAL:")
res_increase = ((high_res - bau_res) / bau_res) * 100
com_increase = ((high_com - bau_com) / bau_com) * 100
print(f"   Residential increase:  {res_increase:.1f}%")
print(f"   Commercial increase:   {com_increase:.1f}%")

print(f"\n3. BUILDING COVERAGE:")
bau_upgraded = (bau_results['upgrade_required'] == 1).sum()
high_upgraded = (high_results['upgrade_required'] == 1).sum()
print(f"   BAU - Buildings requiring upgrades: {bau_upgraded:,} ({bau_upgraded/len(bau_results)*100:.1f}%)")
print(f"   High - Buildings requiring upgrades: {high_upgraded:,} ({high_upgraded/len(high_results)*100:.1f}%)")

print(f"\n4. REGIONAL VARIATION:")
print(f"   California shows the highest costs in both scenarios")
ca_pct_bau = ca_bau / bau_total * 100
ca_pct_high = ca_high / high_total * 100
print(f"   CA represents {ca_pct_bau:.1f}% of BAU costs and {ca_pct_high:.1f}% of High costs")

print("\n" + "="*70)


KEY FINDINGS: SCENARIO COMPARISON

1. COST IMPACT:
   The High scenario requires $2,793,570,358 more in annual costs
   This represents a 47.3% increase over BAU

2. RESIDENTIAL VS COMMERCIAL:
   Residential increase:  48.8%
   Commercial increase:   34.3%

3. BUILDING COVERAGE:
   BAU - Buildings requiring upgrades: 119,278 (13.5%)
   High - Buildings requiring upgrades: 172,948 (19.6%)

4. REGIONAL VARIATION:
   California shows the highest costs in both scenarios
   CA represents 17.7% of BAU costs and 12.4% of High costs



## Implications

### Policy Considerations

1. **Infrastructure Planning**: High scenario requires significantly more grid infrastructure investment
2. **Cost Distribution**: Impacts fall unevenly across regions
3. **Timeline**: Early action is needed to spread costs and avoid peak period congestion
4. **Technology Mix**: Different technologies have different infrastructure requirements

### Next Steps

- Review the [Data Requirements](data-requirements.html) notebook to understand technology adoption patterns
- Explore the [Custom Distributions](custom-distributions.html) notebook to learn about cost variations
- Check the [API Reference](../api-reference.html) for custom analysis methods